In [4]:
import os
# Replace with your actual UUID
os.environ["CUDA_VISIBLE_DEVICES"] = "MIG-822aef03-bf94-5d72-bd27-dd86770c43e9"

In [1]:
import numpy as np
from PIL import Image

# matplotlib inline
%matplotlib inline

def visualize_image_annotations(image_path, txt_path, label_map):
    """
    Visualizes YOLO annotations (bounding boxes and class labels) on an image.

    This function reads an image and its corresponding annotation file in YOLO format, then
    draws bounding boxes around detected objects and labels them with their respective class names.
    The bounding box colors are assigned based on the class ID, and the text color is dynamically
    adjusted for readability, depending on the background color's luminance.

    Args:
        image_path (str): The path to the image file to annotate, and it can be in formats supported by PIL (e.g., .jpg, .png).
        txt_path (str): The path to the annotation file in YOLO format, that should contain one line per object with:
                        - class_id (int): The class index.
                        - x_center (float): The X center of the bounding box (relative to image width).
                        - y_center (float): The Y center of the bounding box (relative to image height).
                        - width (float): The width of the bounding box (relative to image width).
                        - height (float): The height of the bounding box (relative to image height).
        label_map (dict): A dictionary that maps class IDs (integers) to class labels (strings).

    Example:
        >>> label_map = {0: "cat", 1: "dog", 2: "bird"}  # It should include all annotated classes details
        >>> visualize_image_annotations("path/to/image.jpg", "path/to/annotations.txt", label_map)
    """
    import matplotlib.pyplot as plt

    from ultralytics.utils.plotting import colors

    img = np.array(Image.open(image_path))
    img_height, img_width = img.shape[:2]
    annotations = []
    with open(txt_path) as file:
        for line in file:
            class_id, x_center, y_center, width, height = map(float, line.split()[:5])
            x = (x_center - width / 2) * img_width
            y = (y_center - height / 2) * img_height
            w = width * img_width
            h = height * img_height
            annotations.append((x, y, w, h, int(class_id)))
    fig, ax = plt.subplots(1)  # Plot the image and annotations
    for x, y, w, h, label in annotations:
        color = tuple(c / 255 for c in colors(label, True))  # Get and normalize the RGB color
        rect = plt.Rectangle((x, y), w, h, linewidth=1, edgecolor=color, facecolor="none")  # Create a rectangle
        ax.add_patch(rect)
        luminance = 0.2126 * color[0] + 0.7152 * color[1] + 0.0722 * color[2]  # Formula for luminance
        # ax.text(x, y - 5, label_map[label], color="white" if luminance < 0.5 else "black", backgroundcolor=color)
    ax.imshow(img)
    plt.show()


In [ ]:
visualize_image_annotations(
    image_path="datasets/lab-char-detect/images/nlvnpf-0023-022.jpg",
    txt_path="datasets/lab-char-detect/labels/nlvnpf-0023-022.txt",
    output_path="check2.jpg",
    label_map={0: "character"},
)


In [3]:
from ultralytics import YOLO

model = YOLO("/home/projectg7/workspace/nom-ocr-project/trac/runs/detect/lab-runs/tkh-mth2k2-yolo11s-1280-2/weights/best.pt")
results = model("/home/projectg7/datasets/tkh-mth2k2/MTH1000/images/01-V001P000C.png", conf=0.8)

for result in results:
    annotated_image = result.plot(conf=False, line_width=1, font_size=0.1)
    result.show()

ModuleNotFoundError: No module named 'ultralytics'

In [1]:
import glob, os, random, shutil

random.seed(0)
images = sorted(glob.glob("datasets/lab-char-detect/images/*.jpg"))
random.shuffle(images)

n_val = max(1, int(0.2 * len(images)))
splits = {"val": images[:n_val], "train": images[n_val:]}

for split, split_images in splits.items():
    img_out = f"datasets/lab-char-detect/images/{split}"
    lbl_out = f"datasets/lab-char-detect/labels/{split}"
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)
    for img_path in split_images:
        stem = os.path.splitext(os.path.basename(img_path))[0]
        shutil.copy(img_path, f"{img_out}/{stem}.jpg")
        shutil.copy(f"datasets/lab-char-detect/labels_single/{stem}.txt", f"{lbl_out}/{stem}.txt")

print({k: len(v) for k, v in splits.items()})



PermissionError: [Errno 13] Permission denied: 'datasets/lab-char-detect/images/val/nlvnpf-0023-041.jpg'

In [ ]:
visualize_image_annotations(
    "nom-ocr/nomnaocr/Pages/Luc Van Tien/imgs/nlvnpf-0059-002.jpg",  # Input image path.
    "nom-detection/yolo11n-128010/labels/nlvnpf-0059-002.txt",  # Annotation file path for the image.
    # "nom-detection/yolo11n-12807/labels/nlvnpf-0059-002.txt",  # Annotation file path for the image.
    {0: "nom"}  # Class ID to label mapping.,
)
